# OH Target Point Diagnosis
Visualize candidate target points for OH surrogate model using maximum second derivative approach.

In [ ]:
import cantera as ct
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

YAML_FILE = 'chem_cti_toy_model_og.yaml'
mol_units = ct.UnitSystem({
    "length": "cm", "mass": "g", "time": "s",
    "quantity": "mol", "pressure": "dyn / cm^2", "energy": "erg",
    "temperature": "K", "current": "A", "activation-energy": "cal / mol"})

T_INITIAL = 1192
P_INITIAL = 1.95 * ct.one_atm
INITIAL_X = {
    'H2O2': 2216e-6,
    'H2O':  1364e-6,
    'O2':   682e-6,
    'AR':   1.0 - (2216 + 1364 + 682) * 1e-6,
}

DT_FINE  = 1e-7
N_FINE   = 10000
T_FINE   = np.linspace(DT_FINE, DT_FINE * N_FINE, N_FINE)
SMOOTH_WIN = 51

print('Setup complete')

In [ ]:
def run_nominal_fine():
    local_gas = ct.Solution(YAML_FILE)
    local_gas.TPX = T_INITIAL, P_INITIAL, INITIAL_X
    reactor = ct.IdealGasReactor(local_gas, energy='on')
    net = ct.ReactorNet([reactor])
    oh_idx = local_gas.species_index('OH')
    profile = np.empty(N_FINE)
    for k in range(N_FINE):
        net.advance(T_FINE[k])
        profile[k] = reactor.thermo.X[oh_idx]
    return profile

def find_time_for_value(profile, t_sim, target_value, search_start_idx=0):
    search_region = profile[search_start_idx:]
    if search_region.size == 0:
        return None, None
    crossing_mask = np.abs(search_region - target_value) < np.max(profile) * 0.01
    if not crossing_mask.any():
        return None, None
    idx_in_region = np.argmin(np.abs(search_region - target_value))
    actual_idx = search_start_idx + idx_in_region
    return t_sim[actual_idx], actual_idx

def find_inflection(profile, t_sim, i_peak):
    prof_d = profile[i_peak:].astype(float)
    t_d = t_sim[i_peak:]
    w = min(SMOOTH_WIN, max(5, 2 * (len(prof_d) // 100) + 1))
    if w % 2 == 0:
        w += 1
    smooth = uniform_filter1d(prof_d, size=w)
    d2 = np.gradient(np.gradient(smooth, t_d), t_d)
    skip = max(3, len(d2) // 50)
    d2_sub = d2[skip:]
    crossings = np.where(np.diff(np.sign(d2_sub)))[0]
    if crossings.size > 0:
        i_infl_rel = skip + crossings[0] + 1
    else:
        third = max(1, len(d2_sub) // 3)
        i_infl_rel = skip + int(np.argmax(np.abs(d2_sub[:third])))
    i_infl = i_peak + min(i_infl_rel, len(prof_d) - 1)
    return t_sim[min(i_infl, len(t_sim) - 1)], i_infl

print('Running nominal OH profile...')
profile = run_nominal_fine()
print('Profile computed')

In [ ]:
i_peak = int(np.argmax(profile))
t_peak = T_FINE[i_peak]
oh_peak = float(profile[i_peak])
oh_init = float(profile[0])
oh_final = float(profile[-1])

print('='*70)
print('OH Profile Summary')
print('='*70)
print(f'Initial OH: {oh_init*1e6:.2f} ppm (at t=0)')
print(f'Peak OH: {oh_peak*1e6:.2f} ppm (at t={t_peak*1e3:.4f} ms)')
print(f'Gap (peak-init): {(oh_peak-oh_init)*1e6:.2f} ppm')
print(f'Final OH: {oh_final*1e6:.2f} ppm (at t={T_FINE[-1]*1e3:.4f} ms)')
print()

## Maximum Second Derivative Approach

Find: max(y'') where y'' = d²OH/dt²

**Why:** During decay after peak, y'' < 0. At kink point, y'' reaches maximum (least negative) where slope transitions from steep to gradual.

In [ ]:
print('\n' + '='*70)
print('CALCULATING CANDIDATES')
print('='*70)

candidates = {}

print('\n1. PRE-PEAK (average of initial and peak on rising edge)')
oh_prepeak = (oh_init + oh_peak) / 2.0
print(f'   Target value: {oh_prepeak*1e6:.2f} ppm')

mask_rising = T_FINE < t_peak
if mask_rising.any():
    profile_rising = profile[mask_rising]
    t_rising = T_FINE[mask_rising]
    idx_closest = int(np.argmin(np.abs(profile_rising - oh_prepeak)))
    t_prepeak = t_rising[idx_closest]
    i_prepeak = int(np.where(T_FINE == t_rising[idx_closest])[0][0])
    print(f'   FOUND at t={t_prepeak*1e3:.4f} ms')
    candidates['Pre-peak (avg init+peak)'] = (oh_prepeak, t_prepeak, i_prepeak)
else:
    print('   NOT FOUND')
    candidates['Pre-peak (avg init+peak)'] = (oh_prepeak, None, None)

print('\n2. PEAK')
print(f'   FOUND at t={t_peak*1e3:.4f} ms, OH={oh_peak*1e6:.2f} ppm')
candidates['Peak'] = (oh_peak, t_peak, i_peak)

print('\n3. INFLECTION POINT (computed first for kink window)')
t_inflection, i_inflection = find_inflection(profile, T_FINE, i_peak)
oh_inflection = profile[i_inflection]
print(f'   FOUND at t={t_inflection*1e3:.4f} ms, OH={oh_inflection*1e6:.2f} ppm')
candidates['Inflection point'] = (oh_inflection, t_inflection, i_inflection)

print('\n4. KINK (maximum absolute curvature between peak and inflection)')
dt_arr = np.gradient(T_FINE)
doh_dt = np.gradient(profile, dt_arr)
d2oh_dt2 = np.gradient(doh_dt, dt_arr)
w_smooth = min(51, max(5, len(d2oh_dt2) // 100 + 1))
if w_smooth % 2 == 0:
    w_smooth += 1
d2oh_dt2_smooth = uniform_filter1d(d2oh_dt2, size=w_smooth)

# Search between peak and inflection point
i_start = i_peak + 1
i_end = i_inflection
d2_region = np.abs(d2oh_dt2_smooth[i_start:i_end])

print(f'   Search window: i={i_start} to {i_end} (from peak to inflection, {len(d2_region)} points)')

if len(d2_region) > 0:
    i_kink_rel = int(np.argmax(d2_region))
    i_kink = i_start + i_kink_rel
    t_kink = T_FINE[i_kink]
    oh_kink = profile[i_kink]
    print(f'   FOUND at t={t_kink*1e3:.4f} ms, OH={oh_kink*1e6:.2f} ppm')
    print(f'   (at index {i_kink}, curvature magnitude: {d2_region[i_kink_rel]:.6e})')
    candidates['Kink (max |y\'\'|)'] = (oh_kink, t_kink, i_kink)
else:
    print('   NOT FOUND (empty window)')
    candidates['Kink (max |y\'\'|)'] = (None, None, None)

print('\n5-6. DECAY PERCENTILES')
gap = oh_peak - oh_init
print(f'   Gap: {gap*1e6:.2f} ppm')
for pct in [30, 50]:
    oh_target = oh_peak - (pct / 100.0) * gap
    t_target, i_target = find_time_for_value(profile, T_FINE, oh_target, search_start_idx=i_peak)
    label = f'{pct}% below peak'
    if t_target is not None and t_target > 0:
        print(f'   {label}: FOUND at t={t_target*1e3:.4f} ms, OH={oh_target*1e6:.2f} ppm')
    else:
        print(f'   {label}: NOT FOUND')
    candidates[label] = (oh_target, t_target, i_target)

In [ ]:
print('\n' + '='*70)
print('SUMMARY TABLE')
print('='*70)
print(f'{"Label":<30} {"OH (ppm)":>12} {"Time (ms)":>12} {"Status":>12}')
print('-'*70)

for label, (oh_val, t_val, i_val) in candidates.items():
    if t_val is not None and t_val > 0:
        status = 'FOUND'
        time_str = f'{t_val*1e3:.4f}'
    else:
        status = 'NOT FOUND'
        time_str = '—'
    print(f'{label:<30} {oh_val*1e6:>12.2f} {time_str:>12} {status:>12}')

print()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(T_FINE * 1e3, profile * 1e6, 'b-', linewidth=2.5, label='OH profile', zorder=1)
ax.axhline(oh_peak * 1e6, color='r', linestyle='--', alpha=0.5, linewidth=1.5, label=f'Peak: {oh_peak*1e6:.1f} ppm')
ax.axhline(oh_init * 1e6, color='g', linestyle='--', alpha=0.5, linewidth=1.5, label=f'Initial: {oh_init*1e6:.1f} ppm')

colors = plt.cm.tab10(np.linspace(0, 1, len(candidates)))
for idx, (label, (oh_val, t_val, i_val)) in enumerate(candidates.items()):
    if t_val is not None and t_val > 0:
        ax.axhline(oh_val * 1e6, color=colors[idx], linestyle=':', alpha=0.4, linewidth=1)
        ax.plot(t_val * 1e3, oh_val * 1e6, 'o', color=colors[idx], markersize=11,
                label=f'{label} ({oh_val*1e6:.1f} ppm @ {t_val*1e3:.4f} ms)', zorder=5)

ax.set_xlabel('Time (ms)', fontsize=12, fontweight='bold')
ax.set_ylabel('OH mole fraction (ppm)', fontsize=12, fontweight='bold')
ax.set_title('OH Profile with Candidate Target Points', fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=9, framealpha=0.95)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('oh_target_candidates.png', dpi=150, bbox_inches='tight')
print('Saved: oh_target_candidates.png')
plt.show()